# Kitsune Dataset — Cross-Category Ensemble Evaluation

This notebook performs a **cross-category evaluation** of the Hybrid Intrusion Ensemble on the Kitsune dataset.

Unlike the per-attack-type approach (where separate models are trained for each binary subset), here we:
1. **Train once** — a single RF and a single AE on the full combined dataset (all 4 attack types vs. Normal).
2. **Evaluate separately per attack type** — measuring how well each model *generalises* to detect each specific threat.

This cross-category view reveals which attack types the ensemble struggles with when it has never been given a dedicated binary signal for them, and whether the ensemble's two-component design compensates for individual weaknesses.

The Kitsune dataset has 115 numeric features and a `category` column (0–4):
- **0** = Benign (Normal)
- **1** = Reconnaissance
- **2** = Man in the Middle
- **3** = Denial of Service
- **4** = Botnet Malware

## 1. Imports and Configuration

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, precision_recall_curve
)

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

from src.autoencoder.autoencoder import Autoencoder
from src.ensemble.hybrid_ensemble import HybridIntrusionEnsemble

# Styling
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Paths
KITSUNE_PATH = str(project_root / 'data' / 'kitsune_2M.csv')

# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print('\u2713 Imports complete')

## 2. Helper Functions

In [ ]:
def evaluate_model(y_true, y_pred, y_proba, name="Model", verbose=True):
    """Calculate and optionally print metrics."""
    if y_proba.ndim > 1:
        y_proba = y_proba[:, 1]

    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1_score': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_proba)
    }

    if verbose:
        print(f"\n\U0001F4CA {name}:")
        for k, v in metrics.items():
            print(f"  {k:12s}: {v:.4f}")

    return metrics


def plot_comparison(models_data, dataset_name="Dataset"):
    """Plot ROC curves and metric comparison."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # ROC Curves
    ax = axes[0]
    colors = ['steelblue', 'forestgreen', 'purple', 'orange']

    for (name, data), color in zip(models_data.items(), colors):
        y_true = data['y_true']
        y_proba = data['y_proba'][:, 1] if data['y_proba'].ndim > 1 else data['y_proba']

        fpr, tpr, _ = roc_curve(y_true, y_proba)
        auc = data['metrics']['roc_auc']

        ax.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc:.4f})', color=color)

    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(f'ROC Curves \u2014 {dataset_name}', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

    # Metric Comparison
    ax = axes[1]
    metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    metric_keys = ['accuracy', 'precision', 'recall', 'f1_score']

    x = np.arange(len(metric_names))
    width = 0.8 / len(models_data)

    for i, ((name, data), color) in enumerate(zip(models_data.items(), colors)):
        values = [data['metrics'][k] for k in metric_keys]
        offset = (i - len(models_data)/2 + 0.5) * width
        bars = ax.bar(x + offset, values, width, label=name, color=color, alpha=0.8)

        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=8)

    ax.set_ylabel('Score', fontsize=12)
    ax.set_title(f'Performance Metrics \u2014 {dataset_name}', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metric_names)
    ax.legend()
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()


def print_confusion_matrix(y_true, y_pred, name="Model"):
    """Print confusion matrix in readable format."""
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n{name} \u2014 Confusion Matrix:")
    print(f"  TN={cm[0,0]:6d}  FP={cm[0,1]:6d}")
    print(f"  FN={cm[1,0]:6d}  TP={cm[1,1]:6d}")


print('\u2713 Helper functions defined')

---
## 3. Load and Preprocess Kitsune Data

We load the full 2M-row Kitsune dataset and create a **global 60/20/20 split**.
The split preserves the proportion of each category (stratified on the multi-class label).

In [ ]:
print("Loading Kitsune dataset...")
df = pd.read_csv(KITSUNE_PATH)

print(f"\nDataset shape: {df.shape}")
print(f"Features: {df.shape[1] - 1}")
print(f"\nCategory distribution:")
print(df['category'].value_counts().sort_index())

ATTACK_TYPES = {
    1: 'Reconnaissance',
    2: 'Man in the Middle',
    3: 'Denial of Service',
    4: 'Botnet Malware'
}

X_all = df.drop(columns=['category']).values
y_all = df['category'].values  # multi-class: 0=Normal, 1-4=attacks
INPUT_DIM = X_all.shape[1]  # 115

# ----------------------------------------------------------------
# Global stratified 60/20/20 split (preserves category proportions)
# ----------------------------------------------------------------
X_temp, X_test_full, y_temp, y_test_full = train_test_split(
    X_all, y_all, test_size=0.20, random_state=42, stratify=y_all
)
X_train_full, X_val_full, y_train_full, y_val_full = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp  # 0.25 * 0.80 = 0.20
)

print(f"\nGlobal split:")
print(f"  Train: {len(X_train_full):,}  |  Val: {len(X_val_full):,}  |  Test: {len(X_test_full):,}")
print(f"\nFeature dimensions: {INPUT_DIM}")
print("\u2713 Kitsune dataset loaded")

---
## 4. Global Scaling

A single `StandardScaler` is fit on the **normal (benign) portion of the training set**.
This ensures the AE sees reconstruction errors that are meaningful across all attack types.

In [ ]:
# Fit scaler only on normal training samples
normal_train_mask = (y_train_full == 0)
X_normal_train = X_train_full[normal_train_mask]

scaler = StandardScaler()
scaler.fit(X_normal_train)

# Transform all splits
X_train_scaled = scaler.transform(X_train_full)
X_val_scaled   = scaler.transform(X_val_full)
X_test_scaled  = scaler.transform(X_test_full)

print(f"Normal training samples used to fit scaler: {X_normal_train.shape[0]:,}")
print("\u2713 Global scaler fitted")

---
## 5. Train Global Autoencoder (on Normal Traffic Only)

The AE is trained to reconstruct benign traffic. Any input that produces a high reconstruction
error is therefore flagged as anomalous — regardless of which specific attack type it is.

In [ ]:
print(f"Training global AE ({INPUT_DIM} \u2192 32 \u2192 {INPUT_DIM})...")

ae_model = Autoencoder(input_dim=INPUT_DIM, latent_dim=32).to(device)
optimizer = torch.optim.Adam(ae_model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()

# Only normal samples for AE training
X_ae_train = X_train_scaled[normal_train_mask]

EPOCHS = 50
BATCH_SIZE = 2048

ae_model.train()
for epoch in range(1, EPOCHS + 1):
    perm = np.random.permutation(len(X_ae_train))
    epoch_loss = 0.0
    n_batches = 0
    for start in range(0, len(X_ae_train), BATCH_SIZE):
        batch = torch.from_numpy(X_ae_train[perm[start:start+BATCH_SIZE]]).float().to(device)
        optimizer.zero_grad()
        recon = ae_model(batch)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    if epoch % 10 == 0:
        print(f"  Epoch {epoch:3d}/{EPOCHS}, Loss: {epoch_loss/n_batches:.6f}")

ae_model.eval()
print("\u2713 Global AE trained")

---
## 6. Train Global Random Forest (Binary: Normal vs. Any Attack)

The RF is trained on a **binary** version of the full dataset:  
- `0` = Normal (category 0)  
- `1` = Attack (categories 1–4 merged)

This mirrors the AE's perspective (anomaly vs. normal) and feeds the ensemble with a consistent signal.

In [ ]:
# Binarise labels: 0 = Normal, 1 = Any attack
y_train_bin = (y_train_full != 0).astype(int)
y_val_bin   = (y_val_full   != 0).astype(int)
y_test_bin  = (y_test_full  != 0).astype(int)

print("Training global Random Forest (Normal vs. All Attacks)...")
print(f"  Train positives (attacks): {y_train_bin.sum():,} / {len(y_train_bin):,}")

rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_scaled, y_train_bin)
print("\u2713 Global RF trained")

---
## 7. Calibrate Global Ensemble

The `HybridIntrusionEnsemble` combines RF and AE scores. It needs to know the range of AE
reconstruction errors on normal traffic (fitted on the validation set) so it can normalise them.

In [ ]:
ensemble = HybridIntrusionEnsemble(
    rf_model=rf,
    ae_model=ae_model,
    rf_weight=0.7,
    ae_weight=0.3,
    threshold=0.3,
    device=device
)

# Calibrate on normal validation samples only
X_normal_val = X_val_scaled[y_val_bin == 0]
ensemble.fit(X_normal_val)

print(f"\u2713 Global Ensemble calibrated on {len(X_normal_val):,} normal val samples")
print(f"  AE error range: [{ensemble.ae_min_error_:.6f}, {ensemble.ae_max_error_:.6f}]")

---
## 8. Cross-Category Evaluation

Now we evaluate the **same global models** against each attack type individually.  
For each attack category `c`, we build a test subset containing all Normal samples from the test
set **plus** all samples of category `c`, re-label them as 0/1, and measure performance.

This answers: *"When the model sees Normal traffic mixed with only Reconnaissance (or DoS, etc.),
how well does it detect the threat?"* — without ever having been tuned specifically for that binary task.

In [ ]:
all_results = []

# Normal test samples (reused across all attack-type evaluations)
normal_test_mask = (y_test_full == 0)
X_test_normal = X_test_scaled[normal_test_mask]
y_test_normal_bin = np.zeros(X_test_normal.shape[0], dtype=int)

for attack_id, attack_name in ATTACK_TYPES.items():
    print("="*70)
    print(f"ATTACK TYPE: {attack_name} (category {attack_id})")
    print("="*70)

    # Build binary test subset: Normal vs. this specific attack
    attack_test_mask = (y_test_full == attack_id)
    X_test_attack = X_test_scaled[attack_test_mask]
    y_test_attack_bin = np.ones(X_test_attack.shape[0], dtype=int)

    X_test_sub = np.vstack([X_test_normal, X_test_attack])
    y_test_sub = np.concatenate([y_test_normal_bin, y_test_attack_bin])

    print(f"\n  Test subset size: {len(y_test_sub):,}")
    print(f"    Normal: {y_test_normal_bin.shape[0]:,}")
    print(f"    Attack: {y_test_attack_bin.shape[0]:,}\n")

    # ----------------------------------------------------------------
    # 8a. Random Forest
    # ----------------------------------------------------------------
    y_pred_rf    = rf.predict(X_test_sub)
    y_proba_rf   = rf.predict_proba(X_test_sub)
    rf_metrics   = evaluate_model(y_test_sub, y_pred_rf, y_proba_rf, f"RF ({attack_name})")
    print_confusion_matrix(y_test_sub, y_pred_rf, f"RF ({attack_name})")

    # ----------------------------------------------------------------
    # 8b. Autoencoder
    # ----------------------------------------------------------------
    with torch.no_grad():
        X_tensor = torch.from_numpy(X_test_sub).float().to(device)
        recon    = ae_model(X_tensor)
        ae_errors = torch.mean((recon - X_tensor) ** 2, dim=1).cpu().numpy()

    # Find optimal AE threshold via max-F1 on this subset
    precision_arr, recall_arr, pr_thresholds = precision_recall_curve(y_test_sub, ae_errors)
    f1_scores = np.where(
        (precision_arr + recall_arr) == 0,
        0,
        2 * (precision_arr * recall_arr) / (precision_arr + recall_arr)
    )
    optimal_idx  = np.argmax(f1_scores)
    ae_threshold = pr_thresholds[optimal_idx] if optimal_idx < len(pr_thresholds) else (
        pr_thresholds[-1] if len(pr_thresholds) > 0 else 0.0
    )
    print(f"\n  AE optimal threshold: {ae_threshold:.6f}")

    y_pred_ae  = (ae_errors >= ae_threshold).astype(int)
    ae_metrics = evaluate_model(y_test_sub, y_pred_ae, ae_errors, f"AE ({attack_name})")
    print_confusion_matrix(y_test_sub, y_pred_ae, f"AE ({attack_name})")

    # ----------------------------------------------------------------
    # 8c. Hybrid Ensemble (global, pre-calibrated)
    # ----------------------------------------------------------------
    y_pred_ens  = ensemble.predict(X_test_sub)
    y_proba_ens = ensemble.predict_proba(X_test_sub)
    ens_metrics = evaluate_model(y_test_sub, y_pred_ens, y_proba_ens, f"Ensemble ({attack_name})")
    print_confusion_matrix(y_test_sub, y_pred_ens, f"Ensemble ({attack_name})")

    # ----------------------------------------------------------------
    # 8d. Visualisation
    # ----------------------------------------------------------------
    comparison = {
        'AE': {
            'y_true': y_test_sub, 'y_pred': y_pred_ae,
            'y_proba': ae_errors, 'metrics': ae_metrics
        },
        'RF': {
            'y_true': y_test_sub, 'y_pred': y_pred_rf,
            'y_proba': y_proba_rf, 'metrics': rf_metrics
        },
        'Ensemble': {
            'y_true': y_test_sub, 'y_pred': y_pred_ens,
            'y_proba': y_proba_ens, 'metrics': ens_metrics
        }
    }
    plot_comparison(comparison, f"Kitsune \u2014 Normal vs {attack_name} (cross-category)")

    # Collect for summary
    for model_name, m in [('RF', rf_metrics), ('AE', ae_metrics), ('Ensemble', ens_metrics)]:
        row = {'Attack Type': attack_name, 'Model': model_name}
        row.update(m)
        all_results.append(row)

    print()

---
## 9. Overall Summary

Comparison of all models across all attack types in a single table and bar chart.

In [ ]:
summary_df = pd.DataFrame(all_results)
print("\n" + "="*70)
print("SUMMARY \u2014 Cross-Category Evaluation (all attack types)")
print("="*70)
display(summary_df.style.format({
    'accuracy': '{:.4f}',
    'precision': '{:.4f}',
    'recall': '{:.4f}',
    'f1_score': '{:.4f}',
    'roc_auc': '{:.4f}'
}))

In [ ]:
# Grouped bar chart: F1 score per attack type, grouped by model
fig, ax = plt.subplots(figsize=(12, 6))

attack_names = list(ATTACK_TYPES.values())
model_names  = ['RF', 'AE', 'Ensemble']
colors       = ['steelblue', 'forestgreen', 'purple']

x     = np.arange(len(attack_names))
width = 0.25

for i, (model, color) in enumerate(zip(model_names, colors)):
    f1_vals = summary_df[summary_df['Model'] == model]['f1_score'].values
    offset  = (i - 1) * width
    bars    = ax.bar(x + offset, f1_vals, width, label=model, color=color, alpha=0.85)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('F1 Score', fontsize=13)
ax.set_title(
    'F1 Score by Attack Type and Model \u2014 Cross-Category Evaluation (Kitsune)',
    fontsize=14, fontweight='bold'
)
ax.set_xticks(x)
ax.set_xticklabels(attack_names, fontsize=11)
ax.set_ylim(0, 1.08)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()